# Telugu Typed in Roman Letters

Most Telugu speakers do not type in Telugu script. They type Telugu in the English alphabet:
`meeku ela sahayam cheyyanu`, `nenu bagunnanu`, `mee shop enni gantalaki open avutundi`. A chatbot
that decides its input language by looking at Unicode character ranges sees only Latin letters and
files all of that as English.

This recipe routes such a message through Sarvam's Transliterate endpoint instead, and then measures
where that conversion is imperfect.

Pipeline overview:
1. Convert romanised Telugu into Telugu script with the Transliterate API (`te-IN` to `te-IN`)
2. Answer the Telugu-script question with the Chat Completions API (`sarvam-105b`)
3. Convert the Telugu-script answer back to Roman letters (`te-IN` to `en-IN`) so a reader who
   does not read the script can still read the reply
4. Run 30 real romanised messages through step 1 against hand-written Telugu references and print a
   table of which sounds slip

In [ ]:
%pip install -r requirements.txt

## Setup

In [ ]:
from __future__ import annotations

import csv
import json
import os
from difflib import SequenceMatcher
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI
from sarvamai.core.api_error import ApiError

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## The problem, reproduced

`examples/Multilingual_Chatbot/chatbot.py` decides the language of an incoming message by scanning
its characters against Unicode script ranges (lines 36-57), and returns `"english"` when nothing
matches. The cell below is that same logic, copied out so you can run it here without an API key.

Latin letters are not in any Indic range, so every romanised Telugu message falls through to the
final `return "english"` and gets answered in English.

In [ ]:
# Language detection by Unicode range, as in examples/Multilingual_Chatbot/chatbot.py lines 36-57.
UNICODE_RANGES = {
    "hindi": range(0x0900, 0x097F),
    "tamil": range(0x0B80, 0x0BFF),
    "telugu": range(0x0C00, 0x0C7F),
    "kannada": range(0x0C80, 0x0CFF),
    "malayalam": range(0x0D00, 0x0D7F),
}


def detect_language_by_range(text: str) -> str:
    """Return a language name based on the first Indic character found, else english."""
    for char in text:
        code_point = ord(char)
        for language, code_range in UNICODE_RANGES.items():
            if code_point in code_range:
                return language
    return "english"


for message in [
    "meeru ekkada unnaru",
    "meeku ela sahayam cheyyanu",
    "మీరు ఎక్కడ ఉన్నారు",
    "nenu bagunnanu",
]:
    print(f"{detect_language_by_range(message):8s} <- {message}")

## Step 1: romanised Telugu into Telugu script

Both language codes are `te-IN`. The Transliterate endpoint reads romanised or code-mixed input in
the source language and writes it out in that language's own script, so `te-IN` to `te-IN` is the
correct pair here, not `en-IN` to `te-IN` (that pair would spell the English sounds out in Telugu
letters instead of recovering the Telugu words).

`spoken_form=False` keeps digits and times as written. Set it to `True` when the output is going
straight into text-to-speech and you want `9:30` read out as words.

In [ ]:
def to_telugu_script(text: str) -> str:
    """Convert romanised or code-mixed Telugu into Telugu script."""
    try:
        response = client.text.transliterate(
            input=text,
            source_language_code="te-IN",
            target_language_code="te-IN",
            numerals_format="international",
            spoken_form=False,
        )
    except ApiError as e:
        raise RuntimeError(f"Transliterate request failed for {text!r}: {e}") from e

    return response.transliterated_text

## Step 2: answer the question in Telugu

`reasoning_effort` is disabled because `sarvam-105b` reasons by default and those tokens count
against `max_tokens`, which can truncate a short reply into nothing.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant for Telugu speakers. Reply in Telugu script only, "
    "in one or two short sentences. Keep the wording simple and everyday."
)


def answer_in_telugu(telugu_text: str) -> str:
    """Answer a Telugu-script question with sarvam-105b, in Telugu script."""
    try:
        response = client.chat.completions(
            model="sarvam-105b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": telugu_text},
            ],
            temperature=0.2,
            max_tokens=800,
            reasoning_effort=None,
        )
    except ApiError as e:
        raise RuntimeError(f"Chat Completions request failed: {e}") from e

    return response.choices[0].message.content

## Step 3: the answer back into Roman letters

Someone who types `nenu bagunnanu` may not read Telugu script fluently, so the reply is romanised
before it goes back to them. Here the target is `en-IN`, which means "write these Telugu words with
the English alphabet" rather than "translate this into English".

In [ ]:
def to_roman(telugu_text: str) -> str:
    """Write Telugu-script text out in Roman letters, keeping the Telugu words."""
    try:
        response = client.text.transliterate(
            input=telugu_text,
            source_language_code="te-IN",
            target_language_code="en-IN",
            numerals_format="international",
            spoken_form=False,
        )
    except ApiError as e:
        raise RuntimeError(f"Transliterate request failed for {telugu_text!r}: {e}") from e

    return response.transliterated_text

## The three steps together

One message in, four strings out: what the user typed, what it was read as, what the model said,
and what the user gets back.

In [ ]:
def reply_to_romanised(message: str) -> dict[str, str]:
    """Read a romanised Telugu message, answer it, and reply in Roman letters."""
    telugu_input = to_telugu_script(message)
    telugu_reply = answer_in_telugu(telugu_input)
    roman_reply = to_roman(telugu_reply)
    return {
        "roman_input": message,
        "telugu_input": telugu_input,
        "telugu_reply": telugu_reply,
        "roman_reply": roman_reply,
    }


DEMO_MESSAGE = "mee shop enni gantalaki open avutundi"

demo = reply_to_romanised(DEMO_MESSAGE)
for label, value in demo.items():
    print(f"{label:13s} {value}")

demo_path = OUTPUT_DIR / "demo_reply.json"
demo_path.write_text(json.dumps(demo, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nSaved {demo_path}")

## Measuring step 1

Step 1 is the load-bearing step: if the Telugu script it produces is wrong, the model answers a
different question and the rest of the pipeline is polite nonsense.

Below are 30 messages of the kind people actually send, each paired with the Telugu the writer
meant. The reference is the intended Telugu word, not a mechanical letter-by-letter mapping, because
that is what the chat model needs in order to answer correctly.

The set is built to press on the places where casual romanisation is genuinely ambiguous:

- **Hard and soft consonants.** Telugu distinguishes retroflex from dental. Roman spelling does not,
  and the informal convention is inconsistent: `t` usually means the retroflex sound, `th` usually
  means the dental one, so `prathi` is a plain dental t and not an aspirated one.
- **Aspiration.** `dh` in `dhanyavadalu` is genuinely aspirated; `dh` in `adhi` is not.
- **Doubled consonants.** `ekkada`, `cheppu`, `ninna`, `ammayi`.
- **Word endings.** `cheppu` and `cheppandi` are the same verb, informal and polite. Only the final
  vowel separates them, and getting it wrong changes the register of the whole message.

In [ ]:
# (romanised message, intended Telugu, English gloss)
MESSAGES: list[tuple[str, str, str]] = [
    ("meeru ekkada unnaru", "మీరు ఎక్కడ ఉన్నారు", "where are you"),
    ("nenu bagunnanu", "నేను బాగున్నాను", "I am fine"),
    ("meeku ela sahayam cheyyanu", "మీకు ఎలా సహాయం చెయ్యను", "how do I help you"),
    ("meeru ekkada nunchi vasthunnaru", "మీరు ఎక్కడ నుంచి వస్తున్నారు", "where are you coming from"),
    ("naaku telugu radu", "నాకు తెలుగు రాదు", "I do not know Telugu"),
    ("ee roju vana padutundi", "ఈ రోజు వాన పడుతుంది", "it will rain today"),
    ("naa peru ravi", "నా పేరు రవి", "my name is Ravi"),
    ("meeru enni gantalaki vastaru", "మీరు ఎన్ని గంటలకి వస్తారు", "what time will you come"),
    ("naaku ardham kaledu", "నాకు అర్థం కాలేదు", "I did not understand"),
    ("dhanyavadalu", "ధన్యవాదాలు", "thank you"),
    ("konchem sepu aagandi", "కొంచెం సేపు ఆగండి", "wait a moment"),
    ("naaku dabbulu kavali", "నాకు డబ్బులు కావాలి", "I need money"),
    ("meeru cheppandi", "మీరు చెప్పండి", "you tell me, polite"),
    ("nuvvu cheppu", "నువ్వు చెప్పు", "you tell me, informal"),
    ("adhi naaku telusu", "అది నాకు తెలుసు", "I know that"),
    ("gattiga matladandi", "గట్టిగా మాట్లాడండి", "speak loudly"),
    ("maa amma intiki vellindi", "మా అమ్మ ఇంటికి వెళ్ళింది", "my mother went home"),
    ("repu kaluddam", "రేపు కలుద్దాం", "let us meet tomorrow"),
    ("ninna nenu ravaledu", "నిన్న నేను రాలేదు", "I did not come yesterday"),
    ("meeru bhojanam chesara", "మీరు భోజనం చేశారా", "did you eat"),
    ("shubhodayam", "శుభోదయం", "good morning"),
    ("prathi roju nadichi vellutanu", "ప్రతి రోజు నడిచి వెళ్తాను", "I walk every day"),
    ("naaku ee katha nachindi", "నాకు ఈ కథ నచ్చింది", "I liked this story"),
    ("veedu naa thammudu", "వీడు నా తమ్ముడు", "he is my younger brother"),
    ("akkada emi ledu", "అక్కడ ఏమీ లేదు", "there is nothing there"),
    ("naaku thelusukovali", "నాకు తెలుసుకోవాలి", "I want to know"),
    ("mee number ivvandi", "మీ నంబర్ ఇవ్వండి", "give me your number"),
    ("order status chudandi", "ఆర్డర్ స్టేటస్ చూడండి", "check the order status"),
    ("rendu rojula tarvatha call cheyandi", "రెండు రోజుల తర్వాత కాల్ చేయండి", "call after two days"),
    ("naaku vaddu", "నాకు వద్దు", "I do not want it"),
]

print(f"{len(MESSAGES)} messages in the test set")

## How a mismatch gets labelled

Comparing two strings and reporting a percentage says how much is wrong but not what is wrong. The
code below lines the reference up against the returned text word by word, then character by
character, and puts every difference into one of six buckets.

The Telugu writing system makes this classification straightforward: a hard/soft slip is a single
character swapped for its dental or retroflex partner, aspiration is a swap within a fixed set of
ten pairs, a doubling or cluster error involves the virama (the mark that strips a consonant's
inherent vowel), and an ending error is a vowel sign or anusvara at the tail of a word.

None of this calls the API, so you can read and adjust it before spending any credits.

In [ ]:
VIRAMA = "\u0c4d"      # halant, joins consonants into clusters and doubles them
ANUSVARA = "\u0c02"     # the nasal mark that ends words like sahayam
VOWEL_SIGNS = set("\u0c3e\u0c3f\u0c40\u0c41\u0c42\u0c43\u0c44\u0c46\u0c47\u0c48\u0c4a\u0c4b\u0c4c")

# Retroflex against dental: the distinction Roman spelling loses.
HARD_SOFT_PAIRS = {
    frozenset("టత"),   # hard t  / soft t
    frozenset("డద"),   # hard d  / soft d
    frozenset("ఠథ"),   # hard th / soft th
    frozenset("ఢధ"),   # hard dh / soft dh
    frozenset("ణన"),   # hard n  / soft n
    frozenset("ళల"),   # hard l  / soft l
}

# Unaspirated against aspirated: what an h in the romanised spelling may or may not mean.
ASPIRATION_PAIRS = {
    frozenset("కఖ"),   # k  / kh
    frozenset("గఘ"),   # g  / gh
    frozenset("చఛ"),   # ch / chh
    frozenset("జఝ"),   # j  / jh
    frozenset("టఠ"),   # T  / Th
    frozenset("డఢ"),   # D  / Dh
    frozenset("తథ"),   # t  / th
    frozenset("దధ"),   # d  / dh
    frozenset("పఫ"),   # p  / ph
    frozenset("బభ"),   # b  / bh
}

CATEGORIES = [
    "hard vs soft consonant",
    "aspiration",
    "cluster or doubling",
    "word ending",
    "vowel",
    "other",
]


def _category(expected_seg: str, got_seg: str, near_end: bool) -> str:
    """Label a single difference between the reference and the returned text."""
    if len(expected_seg) == 1 and len(got_seg) == 1:
        pair = frozenset(expected_seg + got_seg)
        if pair in HARD_SOFT_PAIRS:
            return "hard vs soft consonant"
        if pair in ASPIRATION_PAIRS:
            return "aspiration"
    if VIRAMA in expected_seg or VIRAMA in got_seg:
        return "cluster or doubling"

    chars = set(expected_seg) | set(got_seg)
    if near_end and chars & (VOWEL_SIGNS | {ANUSVARA}):
        return "word ending"
    if chars and chars <= VOWEL_SIGNS:
        return "vowel"
    return "other"


def word_diffs(expected: str, got: str) -> list[tuple[str, str, str]]:
    """Return (category, expected fragment, returned fragment) for one word pair."""
    diffs: list[tuple[str, str, str]] = []
    matcher = SequenceMatcher(None, expected, got, autojunk=False)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        near_end = i2 >= len(expected) - 1
        diffs.append((_category(expected[i1:i2], got[j1:j2], near_end), expected[i1:i2], got[j1:j2]))
    return diffs


def message_diffs(expected: str, got: str) -> list[tuple[str, str, str]]:
    """Align two messages word by word, then classify the differences inside each word."""
    expected_words = expected.split()
    got_words = got.split()
    diffs: list[tuple[str, str, str]] = []

    matcher = SequenceMatcher(None, expected_words, got_words, autojunk=False)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        if tag == "replace" and (i2 - i1) == (j2 - j1):
            for offset in range(i2 - i1):
                diffs.extend(word_diffs(expected_words[i1 + offset], got_words[j1 + offset]))
        else:
            # A word was dropped, added, or split; there is no character pairing to classify.
            diffs.append(("other", " ".join(expected_words[i1:i2]), " ".join(got_words[j1:j2])))
    return diffs


def edit_distance(a: str, b: str) -> int:
    """Levenshtein distance in characters, used for the character error rate."""
    if a == b:
        return 0
    previous = list(range(len(b) + 1))
    for i, char_a in enumerate(a, start=1):
        current = [i]
        for j, char_b in enumerate(b, start=1):
            current.append(min(
                previous[j] + 1,
                current[j - 1] + 1,
                previous[j - 1] + (char_a != char_b),
            ))
        previous = current
    return previous[-1]

## Run the test set

One Transliterate call per message, 30 in total. A message that fails is reported and skipped rather
than aborting the run.

In [ ]:
rows: list[dict] = []

for roman, expected, gloss in MESSAGES:
    try:
        predicted = to_telugu_script(roman).strip()
    except RuntimeError as e:
        print(f"skipped {roman!r}: {e}")
        continue

    rows.append({
        "roman": roman,
        "gloss": gloss,
        "expected": expected,
        "predicted": predicted,
        "exact": predicted == expected,
        "distance": edit_distance(expected, predicted),
        "diffs": message_diffs(expected, predicted),
    })

print(f"transliterated {len(rows)} of {len(MESSAGES)} messages")

## The error table

Three numbers and one table. The character error rate is the total edit distance divided by the
total length of the references, so it is comparable across runs even if you change the message set.

Read the table as a map of where to spend effort: the categories with the most diffs are the ones
worth handling before this goes near real users.

In [ ]:
if not rows:
    raise RuntimeError("No messages were transliterated; check SARVAM_API_KEY and your quota.")

exact = sum(1 for row in rows if row["exact"])
total_distance = sum(row["distance"] for row in rows)
total_chars = sum(len(row["expected"]) for row in rows)

print(f"messages tested      : {len(rows)}")
print(f"exact matches        : {exact} of {len(rows)} ({exact / len(rows):.0%})")
print(f"character error rate : {total_distance / total_chars:.1%}")
print()

diff_counts = {name: 0 for name in CATEGORIES}
affected = {name: set() for name in CATEGORIES}
for row in rows:
    for category, _expected_seg, _got_seg in row["diffs"]:
        diff_counts[category] += 1
        affected[category].add(row["roman"])

print(f"{'what slipped':24s} {'diffs':>6s} {'messages':>9s}")
print("-" * 41)
for name in CATEGORIES:
    print(f"{name:24s} {diff_counts[name]:6d} {len(affected[name]):9d}")
print()

print("every mismatch, reference against returned text:")
for row in rows:
    if row["exact"]:
        continue
    pairs = ", ".join(
        f"{expected_seg or '-'} -> {got_seg or '-'} [{category}]"
        for category, expected_seg, got_seg in row["diffs"]
    )
    print(f"  {row['roman']}")
    print(f"    {row['expected']}")
    print(f"    {row['predicted']}")
    print(f"    {pairs}")

## Save the results

The CSV is the artefact worth keeping: rerun it after a model update, or against your own message
log, and diff the two.

In [ ]:
csv_path = OUTPUT_DIR / "transliteration_errors.csv"
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["roman", "gloss", "expected", "predicted", "exact", "distance", "categories"])
    for row in rows:
        writer.writerow([
            row["roman"],
            row["gloss"],
            row["expected"],
            row["predicted"],
            row["exact"],
            row["distance"],
            ";".join(sorted({category for category, _e, _g in row["diffs"]})),
        ])

summary = {
    "messages_tested": len(rows),
    "exact_matches": exact,
    "character_error_rate": round(total_distance / total_chars, 4),
    "diffs_by_category": diff_counts,
}
summary_path = OUTPUT_DIR / "error_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Saved {csv_path}")
print(f"Saved {summary_path}")

## What to do with the result

- **Do not throw the romanised text away.** Keep the original message alongside the Telugu script
  version. When a reply looks wrong, the two side by side tell you immediately whether step 1 or the
  model was at fault.
- **Endings carry politeness.** If the table shows word-ending diffs, that is the category to fix
  first: `cheppu` answered as `cheppandi` is not a spelling error to the person reading it.
- **Short messages are the hard ones.** A single word like `vaddu` has no surrounding context for
  the endpoint to lean on. Longer messages tend to come back cleaner.
- **Detect language properly.** Replace the Unicode-range check with the Transliterate round trip
  shown here, or with a language identification call, before deciding which language to answer in.

Further reading: the [Transliterate API](https://docs.sarvam.ai/api-reference-docs/text/transliterate),
the [Chat Completions API](https://docs.sarvam.ai/api-reference-docs/chat/completions), and the
[Transliterate tutorial](../../getting-started/transliterate/Transliterate_API_Tutorial.ipynb) in this
repository.